In [ ]:
from peft import PeftModel
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
import numpy as np
import random
import os
import shap
import pandas as pd
from nltk.corpus import stopwords
import nltk
import spacy
import re
from sklearn.feature_extraction.text import TfidfVectorizer

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
# Load resources.
nltk.download("stopwords") 
stop_words = set(stopwords.words("english")) 
nlp = spacy.load("en_core_web_sm")

In [ ]:
# Set up constant variables.
INPUT_FOLDER = 'data'
OUTPUT_FOLDER = 'explanations'
DEVICE = 0 if torch.cuda.is_available() else -1

# Make model variables.
MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
PEFT_HEAD = "synth_lora_model_bioclinicalbert"
MAX_LENGTH = 256 
TOP_K = 30 
MIN_TOKEN_FREQ = 10

LABEL_MAP = {
    0: "met",
    1: "unmet"
}

In [ ]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state
RANDOM_STATE = set_random_states(1618)

In [ ]:
# Get real data to work with.
all_data = {}
for folder in os.listdir(f'../../{INPUT_FOLDER}/'):
    if '.' not in folder:
        for sub_folder in os.listdir(f'../../{INPUT_FOLDER}/{folder}'):
            if '.' not in sub_folder and 'T1' in sub_folder:
                for sub_sub_folder in os.listdir(f'../../{INPUT_FOLDER}/{folder}/{sub_folder}'):
                    if '.' not in sub_sub_folder: 
                        daily_nurse = pd.read_excel(f'../../{INPUT_FOLDER}/{folder}/{sub_folder}/{sub_sub_folder}/dailyNurseNotes_{sub_sub_folder.split(' ')[0]}.xlsx')
                        daily_nurse = daily_nurse.dropna(subset=['Note', 'Date', 'Time'])
                        all_data[f"{sub_sub_folder.split(' ')[0]}"] = daily_nurse['Note'].dropna().values.tolist()

In [ ]:
# Load model.
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

model = PeftModel.from_pretrained(base_model, PEFT_HEAD)
tokenizer = AutoTokenizer.from_pretrained(PEFT_HEAD)

# Move model to evaluation mode.
model.eval()

In [ ]:
# Make prediction function.
def predict(texts):

    # SHAP sometimes sends numpy arrays
    if isinstance(texts, np.ndarray):
        texts = texts.tolist()

    # Convert everything to string
    texts = [str(t) for t in texts]

    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.nn.functional.softmax(outputs.logits, dim=1)

    return probs.detach().cpu().numpy()

In [ ]:
for patient, notes in all_data.items():
    # Run explainer.
    explainer = shap.Explainer(predict, tokenizer)

    shap_values = explainer(notes)

    # Extract token scores.
    records = []
    for i, note in enumerate(notes):
        tokens = shap_values.data[i]
        values = shap_values.values[i]

        pred_probs = predict([note])[0]
        pred_class = pred_probs.argmax()

        for token, value in zip(tokens, values):
            records.append({'note_id': i, "token": token, "shap_value": value, "predicted_class": LABEL_MAP[pred_class]})
    df = pd.DataFrame(records)

    # Token normalization.
    def normalize_token(token):
        token = token.lower()
        token.replace('##', '')
        token = re.sub(r"[^a-z]", "", token)
        if len(token) < 3:
            return None
        if token in stop_words:
            return None
        if token == "":
            return None
        doc = nlp(token)
        lemma = doc[0].lemma_
        return lemma

    df['token_clean'] = df['token'].apply(normalize_token)
    df = df.dropna()

    # Rare token removal.
    token_counts = df['token_clean'].value_counts()
    valid_tokens = token_counts[token_counts > MIN_TOKEN_FREQ].index
    df = df[df["token_clean"].isin(valid_tokens)]

    # Extract shap for predicted class converting arrays to floats.
    def extract_scalar(v):
        if isinstance(v, (list, np.ndarray)):
            return float(v[0]) if len(v) > 0 else np.nan
        return float(v)

    df["shap_value"] = df["shap_value"].apply(extract_scalar)

    # TF-IDF Computation
    vectorizer = TfidfVectorizer(
        stop_words = 'english',
        min_df = 3,
        max_df = 0.8
    )

    tfidf_matrix = vectorizer.fit_transform(notes)
    tfidf_vocab = vectorizer.vocabulary_

    def get_tfidf(token):
        if token in tfidf_vocab:
            idx = tfidf_vocab[token]
            return tfidf_matrix[:, idx].mean()
        return 0

    df['tfidf'] = df['token_clean'].apply(get_tfidf)

    # Apply tfidf weighting
    df['weighted_score'] = df['shap_value'] * df['tfidf']

    met_df = df[df['predicted_class'] == 'met']
    unmet_df = df[df['predicted_class'] == 'unmet']

    # Aggregate global token scores.
    unmet_scores = (
        unmet_df.groupby('token_clean')['weighted_score'].mean().sort_values()
    )

    met_scores = (
        met_df.groupby('token_clean')['weighted_score'].mean().sort_values(ascending=False)
    )

    # Get top indicators.
    top_unmet = unmet_scores.head(TOP_K) 
    top_met = met_scores.head(TOP_K)

    # Save results.
    os.makedirs(f'./{OUTPUT_FOLDER}/{patient}')
    top_unmet.to_csv(f"./{OUTPUT_FOLDER}/{patient}/top_unmet_indicators.csv")
    top_met.to_csv(f"./{OUTPUT_FOLDER}/{patient}/top_met_indicators.csv")
    df.to_csv(f"./{OUTPUT_FOLDER}/{patient}/full_token_explanations.csv", index=False)

    # Print summary.
    print("\nTop UNMET indicators:") 
    print(top_unmet) 

    print("\nTop MET indicators:") 
    print(top_met)